# Storage Inventory Analysis

## Objective

This notebook analyzes the storage inventory to identify:

- File size distribution
- Small file issues
- Partition characteristics
- Storage utilization
- Potential optimization opportunities

The findings from this notebook will be used to justify storage optimization decisions in the architecture document.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
DATA_PATH = Path("../data/raw/s3_inventory_sample.csv")

inventory = pd.read_csv(DATA_PATH)

inventory.head()

,key,size_bytes,storage_class,last_modified
0,raw/driver_locations/event_date=2026-05-01/par...,259011353,STANDARD,2026-05-01 02:10:00
1,raw/driver_locations/event_date=2026-05-01/par...,235452654,STANDARD,2026-05-01 02:11:00
2,raw/driver_locations/event_date=2026-05-01/par...,256839754,STANDARD,2026-05-01 02:12:00
3,raw/driver_locations/event_date=2026-05-01/par...,257204984,STANDARD,2026-05-01 02:13:00
4,raw/driver_locations/event_date=2026-05-02/par...,235940891,STANDARD,2026-05-02 02:10:00


In [3]:
inventory.shape

(71, 4)

In [4]:
inventory.columns

Index(['key', 'size_bytes', 'storage_class', 'last_modified'], dtype='object')

In [5]:
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   key            71 non-null     object
 1   size_bytes     71 non-null     int64 
 2   storage_class  71 non-null     object
 3   last_modified  71 non-null     object
dtypes: int64(1), object(3)
memory usage: 2.3+ KB


In [6]:
inventory.describe(include="all")

,key,size_bytes,storage_class,last_modified
count,71,7.100000e+01,71,71
unique,71,NaN,1,71
top,raw/driver_locations/event_date=2026-05-01/par...,NaN,STANDARD,2026-05-01 02:10:00
freq,1,NaN,71,1
mean,NaN,3.886159e+07,NaN,NaN
std,NaN,9.026381e+07,NaN,NaN
min,NaN,1.817730e+05,NaN,NaN
25%,NaN,4.012525e+05,NaN,NaN
50%,NaN,5.482030e+05,NaN,NaN
75%,NaN,7.971710e+05,NaN,NaN


In [7]:
inventory.isnull().sum().sort_values(ascending=False)

key              0
size_bytes       0
storage_class    0
last_modified    0
dtype: int64

In [8]:
inventory.head(10)

,key,size_bytes,storage_class,last_modified
0,raw/driver_locations/event_date=2026-05-01/par...,259011353,STANDARD,2026-05-01 02:10:00
1,raw/driver_locations/event_date=2026-05-01/par...,235452654,STANDARD,2026-05-01 02:11:00
2,raw/driver_locations/event_date=2026-05-01/par...,256839754,STANDARD,2026-05-01 02:12:00
3,raw/driver_locations/event_date=2026-05-01/par...,257204984,STANDARD,2026-05-01 02:13:00
4,raw/driver_locations/event_date=2026-05-02/par...,235940891,STANDARD,2026-05-02 02:10:00
5,raw/driver_locations/event_date=2026-05-02/par...,259575420,STANDARD,2026-05-02 02:11:00
6,raw/driver_locations/event_date=2026-05-02/par...,243834304,STANDARD,2026-05-02 02:12:00
7,raw/driver_locations/event_date=2026-05-02/par...,230831692,STANDARD,2026-05-02 02:13:00
8,raw/driver_locations/event_date=2026-05-08/par...,368073,STANDARD,2026-05-08 00:07:00
9,raw/driver_locations/event_date=2026-05-08/par...,528321,STANDARD,2026-05-08 00:14:00


In [9]:
inventory["key"].head()

0    raw/driver_locations/event_date=2026-05-01/par...
1    raw/driver_locations/event_date=2026-05-01/par...
2    raw/driver_locations/event_date=2026-05-01/par...
3    raw/driver_locations/event_date=2026-05-01/par...
4    raw/driver_locations/event_date=2026-05-02/par...
Name: key, dtype: object

In [11]:
inventory["size_mb"] = inventory["size_bytes"] / (1024 * 1024)

In [12]:
inventory["size_mb"].describe()

count     71.000000
mean      37.061303
std       86.082274
min        0.173352
25%        0.382664
50%        0.522807
75%        0.760242
max      247.550411
Name: size_mb, dtype: float64

In [14]:
inventory.head()

,key,size_bytes,storage_class,last_modified,size_mb
0,raw/driver_locations/event_date=2026-05-01/par...,259011353,STANDARD,2026-05-01 02:10:00,247.012475
1,raw/driver_locations/event_date=2026-05-01/par...,235452654,STANDARD,2026-05-01 02:11:00,224.545149
2,raw/driver_locations/event_date=2026-05-01/par...,256839754,STANDARD,2026-05-01 02:12:00,244.941477
3,raw/driver_locations/event_date=2026-05-01/par...,257204984,STANDARD,2026-05-01 02:13:00,245.289787
4,raw/driver_locations/event_date=2026-05-02/par...,235940891,STANDARD,2026-05-02 02:10:00,225.010768


Histogram:

In [15]:
thresholds = [1, 16, 64, 128]

for threshold in thresholds:
    count = (inventory["size_mb"] < threshold).sum()
    percentage = count / len(inventory) * 100

    print(f"Files < {threshold} MB : {count} ({percentage:.1f}%)")

Files < 1 MB : 60 (84.5%)
Files < 16 MB : 60 (84.5%)
Files < 64 MB : 60 (84.5%)
Files < 128 MB : 60 (84.5%)


In [16]:
inventory.sort_values(
    by="size_mb",
    ascending=False
).head(15)

,key,size_bytes,storage_class,last_modified,size_mb
5,raw/driver_locations/event_date=2026-05-02/par...,259575420,STANDARD,2026-05-02 02:11:00,247.550411
0,raw/driver_locations/event_date=2026-05-01/par...,259011353,STANDARD,2026-05-01 02:10:00,247.012475
3,raw/driver_locations/event_date=2026-05-01/par...,257204984,STANDARD,2026-05-01 02:13:00,245.289787
2,raw/driver_locations/event_date=2026-05-01/par...,256839754,STANDARD,2026-05-01 02:12:00,244.941477
69,raw/driver_locations/event_date=2024-02-15/par...,254471939,STANDARD,2024-02-15 02:00:00,242.683352
68,raw/driver_locations/event_date=2024-01-15/par...,252221244,STANDARD,2024-01-15 02:00:00,240.536922
6,raw/driver_locations/event_date=2026-05-02/par...,243834304,STANDARD,2026-05-02 02:12:00,232.538513
70,raw/driver_locations/event_date=2024-03-15/par...,242347931,STANDARD,2024-03-15 02:00:00,231.120997
4,raw/driver_locations/event_date=2026-05-02/par...,235940891,STANDARD,2026-05-02 02:10:00,225.010768
1,raw/driver_locations/event_date=2026-05-01/par...,235452654,STANDARD,2026-05-01 02:11:00,224.545149


In [17]:
inventory.sort_values(
    by="size_mb"
).head(20)

,key,size_bytes,storage_class,last_modified,size_mb
63,raw/driver_locations/event_date=2026-05-08/par...,181773,STANDARD,2026-05-08 18:32:00,0.173352
26,raw/driver_locations/event_date=2026-05-08/par...,204813,STANDARD,2026-05-08 06:13:00,0.195325
41,raw/driver_locations/event_date=2026-05-08/par...,208941,STANDARD,2026-05-08 11:58:00,0.199262
47,raw/driver_locations/event_date=2026-05-08/par...,220115,STANDARD,2026-05-08 13:40:00,0.209918
17,raw/driver_locations/event_date=2026-05-08/par...,220605,STANDARD,2026-05-08 03:10:00,0.210385
61,raw/driver_locations/event_date=2026-05-08/par...,226542,STANDARD,2026-05-08 18:18:00,0.216047
32,raw/driver_locations/event_date=2026-05-08/par...,252792,STANDARD,2026-05-08 08:55:00,0.241081
15,raw/driver_locations/event_date=2026-05-08/par...,293346,STANDARD,2026-05-08 02:56:00,0.279757
48,raw/driver_locations/event_date=2026-05-08/par...,293668,STANDARD,2026-05-08 13:47:00,0.280064
42,raw/driver_locations/event_date=2026-05-08/par...,300944,STANDARD,2026-05-08 11:05:00,0.287003


Result => Question 2

The storage layout is partitioned by event_date, enabling efficient partition pruning for Athena and Spark workloads.

In [18]:
inventory["storage_class"].value_counts()

STANDARD    71
Name: storage_class, dtype: int64

In [19]:
inventory.sort_values("size_mb").head(15)

,key,size_bytes,storage_class,last_modified,size_mb
63,raw/driver_locations/event_date=2026-05-08/par...,181773,STANDARD,2026-05-08 18:32:00,0.173352
26,raw/driver_locations/event_date=2026-05-08/par...,204813,STANDARD,2026-05-08 06:13:00,0.195325
41,raw/driver_locations/event_date=2026-05-08/par...,208941,STANDARD,2026-05-08 11:58:00,0.199262
47,raw/driver_locations/event_date=2026-05-08/par...,220115,STANDARD,2026-05-08 13:40:00,0.209918
17,raw/driver_locations/event_date=2026-05-08/par...,220605,STANDARD,2026-05-08 03:10:00,0.210385
61,raw/driver_locations/event_date=2026-05-08/par...,226542,STANDARD,2026-05-08 18:18:00,0.216047
32,raw/driver_locations/event_date=2026-05-08/par...,252792,STANDARD,2026-05-08 08:55:00,0.241081
15,raw/driver_locations/event_date=2026-05-08/par...,293346,STANDARD,2026-05-08 02:56:00,0.279757
48,raw/driver_locations/event_date=2026-05-08/par...,293668,STANDARD,2026-05-08 13:47:00,0.280064
42,raw/driver_locations/event_date=2026-05-08/par...,300944,STANDARD,2026-05-08 11:05:00,0.287003


In [20]:
inventory.sort_values(
    "size_mb",
    ascending=False
).head(15)

,key,size_bytes,storage_class,last_modified,size_mb
5,raw/driver_locations/event_date=2026-05-02/par...,259575420,STANDARD,2026-05-02 02:11:00,247.550411
0,raw/driver_locations/event_date=2026-05-01/par...,259011353,STANDARD,2026-05-01 02:10:00,247.012475
3,raw/driver_locations/event_date=2026-05-01/par...,257204984,STANDARD,2026-05-01 02:13:00,245.289787
2,raw/driver_locations/event_date=2026-05-01/par...,256839754,STANDARD,2026-05-01 02:12:00,244.941477
69,raw/driver_locations/event_date=2024-02-15/par...,254471939,STANDARD,2024-02-15 02:00:00,242.683352
68,raw/driver_locations/event_date=2024-01-15/par...,252221244,STANDARD,2024-01-15 02:00:00,240.536922
6,raw/driver_locations/event_date=2026-05-02/par...,243834304,STANDARD,2026-05-02 02:12:00,232.538513
70,raw/driver_locations/event_date=2024-03-15/par...,242347931,STANDARD,2024-03-15 02:00:00,231.120997
4,raw/driver_locations/event_date=2026-05-02/par...,235940891,STANDARD,2026-05-02 02:10:00,225.010768
1,raw/driver_locations/event_date=2026-05-01/par...,235452654,STANDARD,2026-05-01 02:11:00,224.545149


In [21]:
thresholds = [1, 16, 64, 128]

results = []

for threshold in thresholds:
    count = (inventory["size_mb"] < threshold).sum()
    percentage = count / len(inventory) * 100

    results.append({
        "Threshold (MB)": threshold,
        "Files": count,
        "Percentage": round(percentage,2)
    })

pd.DataFrame(results)

,Threshold (MB),Files,Percentage
0,1,60,84.51
1,16,60,84.51
2,64,60,84.51
3,128,60,84.51


In [22]:
inventory["event_date"] = inventory["key"].str.extract(
    r"event_date=([^/]+)"
)

inventory["event_date"].value_counts().sort_index()

2024-01-15     1
2024-02-15     1
2024-03-15     1
2026-05-01     4
2026-05-02     4
2026-05-08    60
Name: event_date, dtype: int64

In [23]:
inventory.groupby("event_date")["size_mb"].agg(
    ["count","mean","median","sum"]
)

,count,mean,median,sum
event_date,,,,
2024-01-15,1,240.536922,240.536922,240.536922
2024-02-15,1,242.683352,242.683352,242.683352
2024-03-15,1,231.120997,231.120997,231.120997
2026-05-01,4,240.447222,245.115632,961.788888
2026-05-02,4,231.309487,228.774641,925.237948
2026-05-08,60,0.499740,0.492630,29.984388


# Findings

## Key Observations

- 84.5% of files are smaller than 1 MB.
- The median file size is 0.52 MB, while the average file size is 37.06 MB. This difference indicates a highly skewed file size distribution.
- Historical partitions contain a small number of large files (~230–247 MB).
- The 2026-05-08 partition contains 60 files with an average size of approximately 0.50 MB.
- The file layout suggests more frequent writes in the 2026-05-08 partition, potentially caused by micro-batch or streaming ingestion.
- All objects in the provided inventory sample use the STANDARD storage class.

## Recommendations

- Implement periodic Apache Iceberg compaction to address the small-file problem.
- Target output file sizes between 128 MB and 512 MB for analytical workloads.
- Prioritize compaction for recently modified partitions with high file counts and small average file sizes.
- Run compaction after high-frequency ingestion workloads where appropriate.
- Configure S3 Lifecycle Policies for historical partitions based on access patterns and retention requirements.
- Monitor file count, median file size and average file size as storage optimization metrics.